# Zero-shot vs Few-shot Comparison

## Zero-shot vs Few-shot Learning in LLMs

Both terms describe how you give an LLM instructions/examples in a prompt to get it to perform a task — without updating the model's weights (no fine-tuning involved).

## Zero-shot

You give the model **only an instruction**, with no examples of what the output should look like. The model relies entirely on what it learned during pretraining to figure out the task.

**Example:**
```
Classify the sentiment of this review as Positive, Negative, or Neutral.

Review: "The food was cold and the service was painfully slow."
Sentiment:
```
The model has never seen a demonstration in this prompt — it just infers from the instruction and its general knowledge that this is a sentiment classification task, and answers: `Negative`.

## Few-shot

You give the model **a handful of examples** (input → output pairs) before asking it to do the same task on a new input. This helps the model understand the exact format, style, or edge-case handling you want, especially for tasks that are ambiguous or unusual.

**Example:**
```
Classify the sentiment of these reviews as Positive, Negative, or Neutral.

Review: "Absolutely loved the ambiance and the staff!"
Sentiment: Positive

Review: "It was okay, nothing special."
Sentiment: Neutral

Review: "Waited an hour and the order was still wrong."
Sentiment: Negative

Review: "The food was cold and the service was painfully slow."
Sentiment:
```
By seeing 3 examples first (this is "3-shot"), the model locks onto the exact label vocabulary (`Positive`/`Negative`/`Neutral`), the format (`Sentiment: <label>`), and the calibration of what counts as "Neutral" vs "Negative" — often improving accuracy over zero-shot, especially for niche or precisely-defined tasks.

## Key differences

| | Zero-shot | Few-shot |
|---|---|---|
| Examples given | 0 | Usually 1–10+ ("one-shot" = 1, "few-shot" = 2+) |
| Relies on | Pretrained general knowledge | Pattern-matching from in-prompt examples |
| Best for | Simple, well-known tasks | Ambiguous formats, unusual labels, style matching |
| Prompt length | Short | Longer (more tokens used) |
| Risk | Model may misinterpret task intent | Model may overfit to quirks of the examples given |

**A quick analogy:** zero-shot is like asking a new employee to "write a polite decline email" and trusting their judgment. Few-shot is like showing them 3 sample decline emails your company has sent before, then asking them to write a new one in the same style.

Both are distinct from **fine-tuning**, where you actually update the model's weights using a training dataset — few-shot prompting achieves something similar "on the fly," purely through context, without any retraining.

## When to use which

**Use Zero-shot when:**
- The task is simple, common, or well-known (e.g., translation, summarization, basic sentiment analysis).
- You want to save tokens/cost and keep prompts short.
- The model already performs well without examples (test it first — modern LLMs are quite good zero-shot).

**Use Few-shot when:**
- The task has a specific/unusual output format you need followed exactly (e.g., custom JSON schema, specific labels like "Urgent/Routine/Spam").
- The task is ambiguous and examples remove guesswork (e.g., "what counts as sarcasm" for your use case).
- You need consistent style/tone matching (e.g., matching a brand's email voice).
- Zero-shot attempts gave wrong or inconsistent results — few-shot is your next fix before considering fine-tuning.

## Simple rule of thumb

1. **Start with zero-shot** — it's cheaper and faster.
2. **If output quality/format is off**, add 2–5 well-chosen few-shot examples covering edge cases.
3. **If even few-shot isn't reliable enough** at scale, consider fine-tuning instead.

**One-line heuristic:** *Zero-shot for general tasks the model already "knows"; few-shot for tasks needing a specific format, style, or precision that only examples can convey.*

Here are practical code examples using the OpenAI-style API (the same pattern works for Claude, Gemini, etc. — just the client differs).

## Zero-shot Example (Sentiment Classification)

In [1]:
from openai import OpenAI

client = OpenAI()

prompt = """Classify the sentiment of this review as Positive, Negative, or Neutral.

Review: "The food was cold and the service was painfully slow."
Sentiment:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)
# Output: Negative

Negative


## Few-shot Example (Sentiment Classification)


In [2]:
from openai import OpenAI

client = OpenAI()

prompt = """Classify the sentiment of these reviews as Positive, Negative, or Neutral.

Review: "Absolutely loved the ambiance and the staff!"
Sentiment: Positive

Review: "It was okay, nothing special."
Sentiment: Neutral

Review: "Waited an hour and the order was still wrong."
Sentiment: Negative

Review: "The food was cold and the service was painfully slow."
Sentiment:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)
# Output: Negative

Negative


## Few-shot Using the `messages` Structure (Alternative, Cleaner Approach)

Instead of cramming examples into one string, you can use the chat `messages` array itself to simulate a conversation of examples:


In [3]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "system", "content": "Classify sentiment as Positive, Negative, or Neutral."},
    {"role": "user", "content": "Absolutely loved the ambiance and the staff!"},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It was okay, nothing special."},
    {"role": "assistant", "content": "Neutral"},
    {"role": "user", "content": "Waited an hour and the order was still wrong."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "The food was cold and the service was painfully slow."}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

print(response.choices[0].message.content)
# Output: Negative

Negative


This approach mimics real dialogue turns, which some models handle even more reliably than a single long prompt string — useful when you want the "examples" to feel like natural prior exchanges rather than a static block of text.

**Key takeaway in code terms:** zero-shot = 1 instruction + 1 query in the prompt; few-shot = instruction + N example pairs + query, either concatenated in one string or spread across `messages`.

## 1) Build classification task with both approaches 2) Measure accuracy improvement


In [5]:
"""
Zero-shot vs Few-shot Classification Experiment
================================================
Task: Sentiment classification (Positive / Negative / Neutral)

This script is structured so you can drop in a REAL LLM call
(OpenAI, Anthropic, etc.) by replacing `call_llm()`. Everything else
(dataset, prompt building, accuracy measurement) stays the same.

Since no API key is configured in this sandbox, `call_llm()` currently
uses a SIMULATED response function that mimics realistic LLM behavior:
- Zero-shot: good at obvious cases, but struggles with ambiguous/mixed/
  sarcastic sentences and sometimes returns off-format labels.
- Few-shot: uses the example patterns to correctly calibrate "Neutral"
  vs "Negative" and handle tricky phrasing.

Swap in a real API (see REAL API CALL section at the bottom) and the
exact same measurement code will give you real numbers.
"""

import random

random.seed(42)

# -----------------------------
# 1. Labeled test dataset
# -----------------------------
# (text, true_label) — includes easy cases AND tricky/ambiguous ones
dataset = [
    ("Absolutely fantastic experience, will come back again!", "Positive"),
    ("Terrible service, I want my money back.", "Negative"),
    ("It was fine, nothing special either way.", "Neutral"),
    ("The staff went above and beyond to help us.", "Positive"),
    ("Cold food, slow service, and a rude waiter.", "Negative"),
    ("Not the best, not the worst. Average.", "Neutral"),
    ("Oh great, ANOTHER hour of waiting. Just what I wanted.", "Negative"),  # sarcasm
    ("The ambiance was lovely but the food was mediocre.", "Neutral"),       # mixed
    ("Best meal I've had all year!", "Positive"),
    ("I've seen worse, I guess.", "Neutral"),                                # ambiguous
    ("Completely disappointed, would not recommend.", "Negative"),
    ("Decent value for the price, no complaints.", "Positive"),
    ("The wait was long but the food made up for it.", "Positive"),          # mixed but net positive
    ("Nothing worked as promised, total waste of time.", "Negative"),
    ("It's okay I suppose, wouldn't rush back.", "Neutral"),
    ("Loved every bit of it, exceeded expectations!", "Positive"),
    ("Meh. Wouldn't recommend, wouldn't warn against it either.", "Neutral"),
    ("Rude staff ruined an otherwise good meal.", "Negative"),
    ("Pretty average burger joint, does the job.", "Neutral"),
    ("An unforgettable, wonderful evening with great food.", "Positive"),
]

LABELS = ["Positive", "Negative", "Neutral"]

# -----------------------------
# 2. Prompt builders
# -----------------------------
def build_zero_shot_prompt(text):
    return f"""Classify the sentiment of this review as Positive, Negative, or Neutral.
Respond with a single word only.

Review: "{text}"
Sentiment:"""


FEW_SHOT_EXAMPLES = [
    ("Absolutely loved the ambiance and the staff!", "Positive"),
    ("It was okay, nothing special.", "Neutral"),
    ("Waited an hour and the order was still wrong.", "Negative"),
    ("Oh sure, love paying extra for cold soup.", "Negative"),          # teaches sarcasm handling
    ("Good food, but service was a bit slow.", "Neutral"),              # teaches mixed -> neutral calibration
]

def build_few_shot_prompt(text):
    examples_block = "\n\n".join(
        f'Review: "{ex}"\nSentiment: {label}' for ex, label in FEW_SHOT_EXAMPLES
    )
    return f"""Classify the sentiment of these reviews as Positive, Negative, or Neutral.
Respond with a single word only.

{examples_block}

Review: "{text}"
Sentiment:"""


# -----------------------------
# 3. Simulated LLM call (SWAP THIS with a real API call — see bottom)
# -----------------------------
def call_llm(prompt, mode):
    """
    Simulated model behavior for demo purposes.
    mode: 'zero_shot' or 'few_shot'
    """
    text = prompt.split('Review: "')[-1].split('"')[0]

    # crude keyword-based "ground truth guess" to simulate a real model's reasoning
    lower = text.lower()
    negative_words = ["terrible", "cold", "rude", "disappointed", "worst", "waste", "ruined", "wrong"]
    positive_words = ["fantastic", "lovely", "best", "loved", "wonderful", "great", "good", "beyond"]

    has_neg = any(w in lower for w in negative_words)
    has_pos = any(w in lower for w in positive_words)
    is_sarcastic = lower.startswith("oh") or "just what i wanted" in lower or "oh sure" in lower
    is_mixed = has_neg and has_pos

    if mode == "zero_shot":
        # Zero-shot: no calibration examples -> weaker on sarcasm & mixed/neutral cases
        if is_sarcastic:
            return "Positive"  # misreads sarcasm as literal positive words
        if is_mixed:
            return random.choice(["Positive", "Negative"])  # guesses, skips Neutral
        if has_neg:
            return "Negative"
        if has_pos:
            return "Positive"
        return random.choice(["Positive", "Negative"])  # no Neutral calibration at all

    else:  # few_shot
        # Few-shot: examples taught sarcasm + mixed-sentiment -> Neutral calibration
        if is_sarcastic:
            return "Negative"  # learned from the sarcasm example
        if is_mixed:
            return "Neutral"   # learned from the "good food, slow service" example
        if has_neg:
            return "Negative"
        if has_pos:
            return "Positive"
        return "Neutral"


# -----------------------------
# 4. Run both experiments
# -----------------------------
def run_experiment(mode):
    correct = 0
    results = []
    for text, true_label in dataset:
        prompt = build_zero_shot_prompt(text) if mode == "zero_shot" else build_few_shot_prompt(text)
        pred = call_llm(prompt, mode)
        is_correct = (pred == true_label)
        correct += is_correct
        results.append((text, true_label, pred, is_correct))
    accuracy = correct / len(dataset)
    return accuracy, results


def print_results(mode_name, accuracy, results):
    print(f"\n{'='*70}")
    print(f"{mode_name} RESULTS")
    print(f"{'='*70}")
    for text, true_label, pred, ok in results:
        mark = "✓" if ok else "✗"
        print(f"{mark}  true={true_label:<9} pred={pred:<9} | {text[:55]}")
    print(f"\nAccuracy: {accuracy:.1%}  ({sum(r[3] for r in results)}/{len(results)} correct)")


if __name__ == "__main__":
    zs_acc, zs_results = run_experiment("zero_shot")
    fs_acc, fs_results = run_experiment("few_shot")

    print_results("ZERO-SHOT", zs_acc, zs_results)
    print_results("FEW-SHOT", fs_acc, fs_results)

    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    print(f"Zero-shot accuracy : {zs_acc:.1%}")
    print(f"Few-shot accuracy  : {fs_acc:.1%}")
    print(f"Improvement        : {(fs_acc - zs_acc)*100:+.1f} percentage points")

from openai import OpenAI
client = OpenAI()

def call_llm(prompt, mode):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()



ZERO-SHOT RESULTS
✓  true=Positive  pred=Positive  | Absolutely fantastic experience, will come back again!
✓  true=Negative  pred=Negative  | Terrible service, I want my money back.
✗  true=Neutral   pred=Positive  | It was fine, nothing special either way.
✓  true=Positive  pred=Positive  | The staff went above and beyond to help us.
✓  true=Negative  pred=Negative  | Cold food, slow service, and a rude waiter.
✗  true=Neutral   pred=Positive  | Not the best, not the worst. Average.
✗  true=Negative  pred=Positive  | Oh great, ANOTHER hour of waiting. Just what I wanted.
✗  true=Neutral   pred=Positive  | The ambiance was lovely but the food was mediocre.
✓  true=Positive  pred=Positive  | Best meal I've had all year!
✗  true=Neutral   pred=Negative  | I've seen worse, I guess.
✓  true=Negative  pred=Negative  | Completely disappointed, would not recommend.
✓  true=Positive  pred=Positive  | Decent value for the price, no complaints.
✓  true=Positive  pred=Positive  | The wait was l